# Full-cycle seismic resilience model for distributed energy networks

This notebook is the **public, organized model entry point** for the six-city dataset in this repository. It implements the supplied Pyomo/Gurobi formulation for a distributed energy network (DEN) under seismic disruption and post-hazard restoration.

The repository reorganization changes file paths, city selection, configuration loading, notebook documentation, and stored outputs only. The scientific constraints and parameter values in the supplied model are retained unless explicitly documented in the repository notes.


## 1. Configuration

Choose the city and Monte Carlo settings here. City-specific workbook paths and annual seismic probabilities are read from `../config/seismic_probabilities.csv`.


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from pyomo.environ import (
    Binary, ConcreteModel, Constraint, NonNegativeReals, Objective, Param,
    Set, SolverFactory, Var, minimize
)
from pyomo.opt import TerminationCondition

# -------- User configuration --------
CITY = "harbin"  # beijing | fuzhou | harbin | puer | wuhan | xian
N_MONTE_CARLO = 1000
RANDOM_SEED = None  # set an integer for repeatable scenario generation
# ------------------------------------

def _find_repo_root(start: Path) -> Path:
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "config" / "seismic_probabilities.csv").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate repository root containing config/seismic_probabilities.csv"
    )

REPO_ROOT = _find_repo_root(Path.cwd())
CITY_CONFIG = pd.read_csv(REPO_ROOT / "config" / "seismic_probabilities.csv").set_index("city_id")
if CITY not in CITY_CONFIG.index:
    raise ValueError(f"Unknown CITY={CITY!r}. Choose one of: {sorted(CITY_CONFIG.index)}")

CITY_META = CITY_CONFIG.loc[CITY]
DATA_FILE = REPO_ROOT / str(CITY_META["data_file"])
SEISMIC_PROBABILITIES = {
    "M6": float(CITY_META["M6"]),
    "M7": float(CITY_META["M7"]),
    "M8": float(CITY_META["M8"]),
}
NORMAL_PROBABILITY = float(CITY_META["normal_probability"])

if RANDOM_SEED is not None:
    np.random.seed(RANDOM_SEED)

print(f"City: {CITY_META['city']} | data: {DATA_FILE.relative_to(REPO_ROOT)}")


## 2. Model index sets

Define six buildings (`b1`–`b6`), 168 hourly steps, representative climate/seismic scenarios, Monte Carlo realizations, technologies, and tariff categories.


In [ ]:
m = ConcreteModel(name="dizheng_IES")
m.i = Set(initialize=[f'b{i}' for i in range(1, 7)], doc='building index')
m.j = Set(initialize=m.i, doc='building index copied i')
m.h = Set(initialize=[f'h{i}' for i in range(1, 169)], doc='hours')
m.sr = Set(initialize=[f'sr{i}' for i in range(1, N_MONTE_CARLO + 1)], doc='MonteCarlo times')
m.s = Set(initialize=['sum', 'win', 'mid', 'M6', 'M7','M8'], doc='scenarios')
m.t = Set(initialize=['chp', 'boiler', 'ec', 'ac', 'hp', 'cool_st', 'heat_st', 'ele_st','grid','pv'], doc='technology')
m.t_eff = Set(initialize=['chp', 'boiler',  'ec', 'ac', 'hp', 'cool_st', 'heat_st', 'ele_st','grid','pv'])
m.p = Set(initialize=['grid_buy1','grid_buy2', 'grid_sell', 'NG',], doc='price category')
m.unitCap = Set(initialize=['cool_st', 'heat_st','ele_st'])
m.maint = Set(initialize=['chp', 'boiler',  'ec', 'ac', 'hp', 'cool_st', 'heat_st', 'ele_st','grid','pv'])

## 3. Input data

Read the seven standardized sheets from the selected city workbook and map them into Pyomo parameters. See `../data/README.md` and `../docs/DATA_DICTIONARY.md` for field-level definitions.


In [ ]:
e_demand_dic = pd.read_excel(DATA_FILE, sheet_name='e_dem', header=0, index_col=[0,1])
m.e_demand = Param(m.i, m.s, m.h, initialize=lambda m,i,s,h: e_demand_dic[h][i][s], doc='electric demand per building per scenario per hour')

h_demand_dic = pd.read_excel(DATA_FILE, sheet_name='h_dem', header=0, index_col=[0,1])
m.h_demand = Param(m.i, m.s, m.h, initialize=lambda m,i,s,h: h_demand_dic[h][i][s], doc='heat demand per building per scenario per hour')

c_demand_dic = pd.read_excel(DATA_FILE, sheet_name='c_dem', header=0, index_col=[0,1])
m.c_demand = Param(m.i, m.s, m.h, initialize=lambda m,i,s,h: c_demand_dic[h][i][s], doc='cooling demand per building per scenario per hour')

price_dic = pd.read_excel(DATA_FILE, sheet_name='price', header=0, index_col=[0])
m.price = Param(m.p, m.h, initialize=lambda m,p,h: price_dic[h][p], doc='each price per scenario')

SRI_dic = pd.read_excel(DATA_FILE, sheet_name='SRI', header=0, index_col=[0])
m.SRI = Param(m.s, m.h, initialize=lambda m,s,h: SRI_dic[h][s], doc='SRI per scenario per scenario per hour')

qty_day_dic = pd.read_excel(DATA_FILE, sheet_name='qty_day', header=0, index_col=[0])
m.qty_day = Param(m.s,m.i, initialize=lambda m,s,i: qty_day_dic[s][i], doc='days per scenario')

distance_dic = pd.read_excel(DATA_FILE, sheet_name='dist',header=0, index_col=0)
m.dist = Param(m.i, m.j, initialize=lambda m, i, j: distance_dic[i][j], doc='distance between each two buildings')

## 4. Fixed techno-economic parameters

Define technology efficiencies, capacity bounds, capital and maintenance coefficients, pipe losses, roof-area limits, scenario probabilities, and component fragility probabilities.


In [ ]:

m.pCap_max = Param(m.t, initialize={'chp': 30000, 'boiler':20000, 'ec':30000, 'ac':30000, 'hp': 20000, 'cool_st': 20000, 'heat_st': 20000, 'ele_st':20000, 'grid':20000, 'pv':2000})
m.pStart_limit_chp = Param(initialize=1)
m.pH_to_P = Param(initialize=1.1)
m.pEff = Param(m.t_eff, initialize={'chp': 0.45,'boiler': 0.85,'ec': 4,'ac': 1.2,'hp': 3, 'cool_st': 0.95, 'heat_st': 0.95, 'ele_st':0.95, 'pv': 0.14})
m.pLoss_rate_cool_pipe = Param(initialize=0.05, doc='loss per 1000m')
m.pLoss_rate_heat_pipe = Param(initialize=0.06, doc='loss per 1000m')
m.pArea_roof = Param(m.i, initialize={'b1': 4500,'b2': 2600,'b3': 2000,'b4': 3800,'b5': 5600,'b6': 12000})
m.pCost_unit = Param(m.t, initialize={'chp': 4100, 'boiler':800, 'ec':1000, 'ac':1500, 'hp': 1200, 'cool_st':117.5, 'heat_st':117.5,'ele_st':1500, 'pv':3500})
m.pCost_unit_hpipe = Param(initialize=1600)
m.pCost_unit_cpipe = Param(initialize=1600)
m.CRF1 = Param(initialize=0.103, doc='capital recovery factor based on interest rate of 0.06 and 15 years')
m.CRF2 = Param(initialize=0.073, doc='capital recovery factor based on interest rate of 0.06 and 30 years')
m.CRF3 = Param(initialize=0.085, doc='capital recovery factor based on interest rate of 0.06 and 25 years')
m.pMaint = Param(m.maint, initialize={'chp': 0.03,'boiler': 0.002,'ec': 0.002,'ac': 0.002,'hp': 0.008, 'cool_st': 0.0013, 'heat_st': 0.0013, 'ele_st':0.0023, 'pv':0.0021})
m.Pro = Param(m.s, initialize={'sum': NORMAL_PROBABILITY, 'win': 1, 'mid': 1, 'M6': SEISMIC_PROBABILITIES['M6'], 'M7': SEISMIC_PROBABILITIES['M7'], 'M8': SEISMIC_PROBABILITIES['M8']})

## 5. Seismic scenarios

Define technology-specific fragility probabilities, initialize Monte Carlo storage, and generate stochastic earthquake onset/damage states for `M6`, `M7`, and `M8`.


In [ ]:
m.Pr_chp_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.13,'M8':0.51})
m.Pr_boiler_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.03,'M8':0.12})
m.Pr_ec_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.045,'M8':0.28})
m.Pr_ac_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.04,'M8':0.27})
m.Pr_hp_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.045,'M8':0.28})
m.Pr_cool_st_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.17,'M8':0.66})
m.Pr_ele_st_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.015,'M8':0.21})
m.Pr_grid_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.01, 'M7':0.075,'M8':0.24})
m.Pr_pv_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.001, 'M7':0.23,'M8':0.86})
m.Pr_pipe_damage = Param(m.s, initialize={'sum':0, 'win':0, 'mid':0, 'M6':0.09, 'M7':0.21,'M8':0.40})


In [ ]:
m.damage_hour_chp1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_boiler1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_ec1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_ac1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_hp1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_cool_st1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_ele_st1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_grid1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_pv1 = Param(m.i,m.s,mutable=True,initialize=0)
m.damage_hour_pipe1 = Param(m.i,m.s,mutable=True,initialize=0)

### Monte Carlo result containers

Pre-allocate mutable Pyomo parameters used to retain costs, capacities, dispatch, damage timing, and resilience metrics from each stochastic realization.


In [ ]:
m.ag_chp_e = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_pv_e = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_ec_ele = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_ec_cool = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_hp_ele = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_b_heat = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_ac_heat = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_ac_cool = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_hp_heat = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_ele_cha = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_ele_dis = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_cool_cha = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_cool_dis = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_grid_im = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_grid_ex = Param(m.sr,m.i,m.s,m.h,mutable=True,initialize=0)
m.ag_tr_h = Param(m.sr,m.i,m.j,m.s,m.h,mutable=True,initialize=0)
m.ag_tr_c = Param(m.sr,m.i,m.j,m.s,m.h,mutable=True,initialize=0)
m.ag_derepair_chp = Param(m.sr,m.i,m.s,mutable=True,initialize=0)

m.ag_Disaster_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_results = Param(m.sr,mutable=True, initialize=0)
m.ag_device_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_pipe_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_maint_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_fuel_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_grid_im_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_grid_ex_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_restored_cost = Param(m.sr,mutable=True, initialize=0)
m.ag_EENS = Param(m.sr,mutable=True, initialize=0)
m.ag_EIU = Param(m.sr,mutable=True, initialize=0)

m.ag_chp_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_ec_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_b_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_hp_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_ac_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_pv_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_est_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_cst_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_grid_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_hpipe_restored_Tcost = Param(m.sr,mutable=True, initialize=0)
m.ag_cpipe_restored_Tcost = Param(m.sr,mutable=True, initialize=0)

m.ag_damage_hour_chp = Param(m.sr,m.i,m.s,mutable=True, initialize=0)
m.ag_damage_hour_pv = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_ec = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_boiler = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_ac = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_hp = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_pipe = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_grid = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_ele_st = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  
m.ag_damage_hour_cool_st = Param(m.sr, m.i, m.s, mutable=True, initialize=0)  

m.ag_CHP_emax = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_ec_cmax = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_ac_cmax = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_hp_hmax = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_b_hmax = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_ele_st_max = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_cool_st_max = Param(m.sr,m.i,mutable=True, initialize=0)
m.ag_pv_areamax = Param(m.sr,m.i,mutable=True, initialize=0)



### Scenario sampling

For each Monte Carlo realization, sample one earthquake onset hour and then sample component failures by comparing random draws with the scenario-specific fragility probabilities.


In [ ]:
h_list1= [f'h{i}' for i in range(1, 169)]
m.damage_hour_chp = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_boiler = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_ec = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_ac = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_hp = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_cool_st = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_ele_st = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_grid = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_pv = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
m.damage_hour_pipe = Param(m.sr,m.i,m.s,mutable=True,initialize=0)
for sr in list(m.sr):
    Disaster_hour = np.random.randint(1, 25)
    for i in list(m.i):
        for s in list(m.s):
            if m.Pr_chp_damage[s] > round(np.random.rand(),4):
                m.damage_hour_chp[sr,i,s] = Disaster_hour 
            else:
                m.damage_hour_chp[sr,i,s] = 169
                
            if m.Pr_boiler_damage[s] > round(np.random.rand(),4):
                m.damage_hour_boiler[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_boiler[sr,i,s] = 169

            if m.Pr_ec_damage[s] > round(np.random.rand(),4):
                m.damage_hour_ec[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_ec[sr,i,s] = 169

            if m.Pr_ac_damage[s] > round(np.random.rand(),4):
                m.damage_hour_ac[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_ac[sr,i,s] = 169

            if m.Pr_hp_damage[s] > round(np.random.rand(),4):
                m.damage_hour_hp[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_hp[sr,i,s] = 169
                    
            if m.Pr_cool_st_damage[s] > round(np.random.rand(),4):
                m.damage_hour_cool_st[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_cool_st[sr,i,s] = 169

            if m.Pr_ele_st_damage[s] > round(np.random.rand(),4):
                m.damage_hour_ele_st[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_ele_st[sr,i,s] = 169

            if m.Pr_grid_damage[s] > round(np.random.rand(),4):
                m.damage_hour_grid[sr,i,s]  = Disaster_hour
            else:
                m.damage_hour_grid[sr,i,s]  = 169

            if m.Pr_pv_damage[s] > round(np.random.rand(),4):
                m.damage_hour_pv[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_pv[sr,i,s] = 169
            if m.Pr_pipe_damage[s] > round(np.random.rand(),4):
                m.damage_hour_pipe[sr,i,s] = Disaster_hour
            else:
                m.damage_hour_pipe[sr,i,s] = 169



In [ ]:
m.damage_chp = Param(m.i, m.s, m.h,mutable=True)
m.damage_b = Param(m.i, m.s, m.h, mutable=True)  
m.damage_hp = Param(m.i, m.s, m.h, mutable=True)  
m.damage_pv = Param(m.i, m.s, m.h, mutable=True)  
m.damage_ec = Param(m.i, m.s, m.h, mutable=True)
m.damage_ac = Param(m.i, m.s, m.h, mutable=True)
m.damage_ele_st = Param(m.i, m.s, m.h, mutable=True)
m.damage_cool_st = Param(m.i, m.s, m.h, mutable=True)
m.damage_grid = Param(m.i, m.s, m.h, mutable=True)
m.damage_pipe = Param(m.i,m.j, m.s, m.h, mutable=True)


## 6. Optimization model

Define dispatch, capacity, storage, grid, inter-building network, unmet-demand, restoration, and cost variables and constraints. The following cells preserve the supplied model formulation.


In [ ]:
m.ec_ele = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.hp_ele = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.grid_ex = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.grid_im = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.ele_cha = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.ele_dis = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.CHP_e = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.pv_e = Var(m.i,m.s,m.h,domain=NonNegativeReals)

m.ac_heat = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.tr_h = Var(m.i,m.j,m.s,m.h,domain=NonNegativeReals)
m.hp_heat = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.b_heat = Var(m.i,m.s,m.h,domain=NonNegativeReals)

m.cool_cha = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.tr_c = Var(m.i,m.j,m.s,m.h,domain=NonNegativeReals)
m.ec_cool = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.ac_cool = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.cool_dis = Var(m.i,m.s,m.h,domain=NonNegativeReals)


#### Energy flow balance
def ele_balance_rule1(m, i, s, h):
    return m.e_demand[i,'sum',h] +m.ec_ele[i,'sum',h] +m.hp_ele[i,'sum',h] +m.grid_ex[i,'sum',h] +m.ele_cha[i,'sum',h] == m.ele_dis[i,'sum',h] +m.CHP_e[i,'sum',h] +m.grid_im[i,'sum',h] +m.pv_e[i,'sum',h]
m.ele_balance1 = Constraint(m.i,m.s,m.h, rule=ele_balance_rule1)
def heat_balance_rule1(m,i,s,h):
    return m.h_demand[i,'sum',h] + m.ac_heat[i,'sum',h] + sum(m.tr_h[i,j,'sum',h] for j in m.j)  == m.hp_heat[i,'sum',h] + m.CHP_e[i,'sum',h] *m.pH_to_P + m.b_heat[i,'sum',h] + sum(m.tr_h[j,i,'sum',h]*(1-m.pLoss_rate_heat_pipe) for j in m.j) 
m.heat_balance1 = Constraint(m.i,m.s,m.h, rule=heat_balance_rule1)
def cool_balance_rule1(m,i,s,h):
    return m.c_demand[i,'sum',h] + sum(m.tr_c[i,j,'sum',h] for j in m.j) +m.cool_cha[i,'sum',h] == m.ec_cool[i,'sum',h] + m.ac_cool[i,'sum',h] + sum(m.tr_c[j,i,'sum',h]*(1-m.pLoss_rate_cool_pipe) for j in m.j) +m.cool_dis[i,'sum',h]
m.cool_balance1 = Constraint(m.i,m.s,m.h, rule=cool_balance_rule1)

def ele_balance_rule2(m, i, s, h):
    return m.e_demand[i,'win',h] +m.ec_ele[i,'win',h] +m.hp_ele[i,'win',h] +m.grid_ex[i,'win',h] +m.ele_cha[i,'win',h] == m.ele_dis[i,'win',h] +m.CHP_e[i,'win',h] +m.grid_im[i,'win',h] +m.pv_e[i,'win',h]
m.ele_balance2 = Constraint(m.i,m.s,m.h, rule=ele_balance_rule2)
def heat_balance_rule2(m,i,s,h):
    return m.h_demand[i,'win',h] + m.ac_heat[i,'win',h] + sum(m.tr_h[i,j,'win',h] for j in m.j)  == m.hp_heat[i,'win',h] + m.CHP_e[i,'win',h] *m.pH_to_P + m.b_heat[i,'win',h] + sum(m.tr_h[j,i,'win',h]*(1-m.pLoss_rate_heat_pipe) for j in m.j) 
m.heat_balance2 = Constraint(m.i,m.s,m.h, rule=heat_balance_rule2)
def cool_balance_rule2(m,i,s,h):
    return m.c_demand[i,'win',h] + sum(m.tr_c[i,j,'win',h] for j in m.j) +m.cool_cha[i,'win',h] == m.ec_cool[i,'win',h] + m.ac_cool[i,'win',h] + sum(m.tr_c[j,i,'win',h]*(1-m.pLoss_rate_cool_pipe) for j in m.j) +m.cool_dis[i,'win',h]
m.cool_balance2 = Constraint(m.i,m.s,m.h, rule=cool_balance_rule2)

def ele_balance_rule3(m, i, s, h):
    return m.e_demand[i,'mid',h] +m.ec_ele[i,'mid',h] +m.hp_ele[i,'mid',h] +m.grid_ex[i,'mid',h] +m.ele_cha[i,'mid',h] == m.ele_dis[i,'mid',h] +m.CHP_e[i,'mid',h] +m.grid_im[i,'mid',h] +m.pv_e[i,'mid',h]
m.ele_balance3 = Constraint(m.i,m.s,m.h, rule=ele_balance_rule3)
def heat_balance_rule3(m,i,s,h):
    return m.h_demand[i,'mid',h] + m.ac_heat[i,'mid',h] + sum(m.tr_h[i,j,'mid',h] for j in m.j)  == m.hp_heat[i,'mid',h] + m.CHP_e[i,'mid',h] *m.pH_to_P + m.b_heat[i,'mid',h] + sum(m.tr_h[j,i,'mid',h]*(1-m.pLoss_rate_heat_pipe) for j in m.j)
m.heat_balance3 = Constraint(m.i,m.s,m.h, rule=heat_balance_rule3)
def cool_balance_rule3(m,i,s,h):
    return m.c_demand[i,'mid',h] + sum(m.tr_c[i,j,'mid',h] for j in m.j) +m.cool_cha[i,'mid',h] == m.ec_cool[i,'mid',h] + m.ac_cool[i,'mid',h] + sum(m.tr_c[j,i,'mid',h]*(1-m.pLoss_rate_cool_pipe) for j in m.j) +m.cool_dis[i,'mid',h]
m.cool_balance3 = Constraint(m.i,m.s,m.h, rule=cool_balance_rule3)

In [ ]:
h_list = [i for i in m.h]
def h_1(h:str):
    return h_list[h_list.index(h)-1] if h != 'h1' else h_list[-1]

In [ ]:

m.CHP_emax = Var(m.i,domain=NonNegativeReals)
m.onoff = Var(m.i,m.s,m.h,domain=Binary)
m.start_chp = Var(m.i,m.s,m.h,domain=Binary)
m.derepair_chp = Var(m.i,m.s, domain=Binary)
m.chp_repaired = Var(m.i, m.s, m.h, domain=Binary, initialize=0) 
m.chp_restored_cost = Var(m.i,domain=NonNegativeReals)
m.chp_restored_Tcost = Var(domain=NonNegativeReals)





##### cchp
## chp capacity limit
def chp_limit1(m,i,s,h):
    return m.CHP_e[i,s,h] <=  m.CHP_emax[i]
m.chp_limit1 = Constraint(m.i,m.s,m.h,rule=chp_limit1)
def chp_limit2(m,i):
    return m.CHP_emax[i] <= m.pCap_max['chp']
m.chp_limit2 = Constraint(m.i,rule=chp_limit2)
def chp_limit3(m):
    return m.CHP_emax['b6'] == 0
m.chp_limit3 = Constraint(rule=chp_limit3)
def chp_limit4(m,i,s,h):
    return m.CHP_e[i,s,h] >= (m.onoff[i,s,h]-1)*m.pCap_max['chp']*2 + 0.3*m.CHP_emax[i]
m.chp_limit4 = Constraint(m.i,m.s,m.h,rule=chp_limit4)
### chp climbing limit
def chp_climb_limit1(m,i,s,h):
    return m.CHP_e[i,s,h]-m.CHP_e[i,s,h_1(h)] <= 0.5*m.CHP_emax[i]
m.chp_climb_limit1 = Constraint(m.i,m.s,m.h,rule=chp_climb_limit1)
def chp_climb_limit2(m,i,s,h):
    return m.CHP_e[i,s,h_1(h)]-m.CHP_e[i,s,h] <= 0.5*m.CHP_emax[i]
m.chp_climb_limit2 = Constraint(m.i,m.s,m.h,rule=chp_climb_limit2)
### chp start limit
def chp_start_limit1(m,i,s):
    return sum(m.start_chp[i,s,h] for h in m.h) <= m.pStart_limit_chp
m.chp_start_limit1 = Constraint(m.i,m.s,rule=chp_start_limit1)
def chp_start_limit2(m,i,s,h):
    return m.start_chp[i,s,h] >= m.onoff[i,s,h] - m.onoff[i,s,h_1(h)]
m.chp_start_limit2 = Constraint(m.i,m.s,m.h,rule=chp_start_limit2)
def chp_start_limit3(m,i,s,h):
    return m.start_chp[i,s,h] <= 1 - m.onoff[i,s,h_1(h)]
m.chp_start_limit3 = Constraint(m.i,m.s,m.h,rule=chp_start_limit3)
def chp_start_limit4(m,i,s,h):
    return m.start_chp[i,s,h] <= m.onoff[i,s,h]
m.chp_start_limit4 = Constraint(m.i,m.s,m.h,rule=chp_start_limit4)
 
def chp_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.CHP_e[i,s,h] <=m.pCap_max['chp'] * 2
    elif Disaster_hour <= h_list1.index(h)+1 < 108:
        return m.CHP_e[i,s,h] <= m.damage_chp[i,s,h]* m.pCap_max['chp'] * 2  # 修复期间，产能为0 
    else:
        return m.CHP_e[i,s,h] <= m.damage_chp[i,s,h] * m.pCap_max['chp'] * 2 +m.derepair_chp[i,s] * m.pCap_max['chp'] * 2 #同时考虑设备损坏与修复
m.chp_repair_limit2 = Constraint(m.i,m.s,m.h,rule=chp_repair_limit2)
    
def chp_restored_cost_limit(m,i):
    return m.chp_restored_cost[i] == (m.derepair_chp[i, 'M6'] + m.derepair_chp[i, 'M7'] + m.derepair_chp[i, 'M8']) * 80000
m.chp_restored_cost_limit = Constraint(m.i, rule=chp_restored_cost_limit)

def chp_restored_cost_limit2(m):
    return m.chp_restored_Tcost == sum(m.chp_restored_cost[i] for i in m.i)
m.chp_restored_cost_limit2 = Constraint(rule=chp_restored_cost_limit2)




In [ ]:
m.ec_cmax = Var(m.i,domain=NonNegativeReals)
m.derepair_ec = Var(m.i,m.s, domain=Binary)
m.ec_restored_cost = Var(m.i,domain=NonNegativeReals)

def ec_limit1(m,i):
    return m.ec_cmax[i] <= m.pCap_max['ec']
m.ec_limit1 = Constraint(m.i,rule=ec_limit1)
def ec_limit2(m,i,s,h):
    return m.ec_cool[i,s,h] == m.ec_ele[i,s,h] * m.pEff['ec']
m.ec_limit2 = Constraint(m.i,m.s,m.h,rule=ec_limit2)
def ec_limit3(m,i,s,h):
    return m.ec_cool[i,s,h] <= m.ec_cmax[i]
m.ec_limit3 = Constraint(m.i,m.s,m.h,rule=ec_limit3)

def ec_not_run_limit1(m,i,h):
    return m.ec_cool[i,'win',h] == 0
m.ec_not_run_limit1 = Constraint(m.i,m.h,rule=ec_not_run_limit1)
def ec_not_run_limit2(m,i,h):
    if i in ['b1', 'b2','b3', 'b4','b6']:
        return m.ec_cool[i,'mid',h] == 0
    else:
        return Constraint.Skip
m.ec_not_run_limit2 = Constraint(m.i,m.h,rule=ec_not_run_limit2)

def ec_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.ec_cool[i,s,h] <=m.pCap_max['ec'] * 2
    elif Disaster_hour <= h_list1.index(h)+1 < 108:
        return m.ec_cool[i,s,h] <= m.damage_ec[i,s,h]* m.pCap_max['ec'] * 2  # 修复期间，产能为0 
    else:
        return m.ec_cool[i,s,h] <= m.damage_ec[i,s,h] * m.pCap_max['ec'] * 2 +m.derepair_ec[i,s] * m.pCap_max['ec'] * 2 #同时考虑设备损坏与修复
m.ec_repair_limit2 = Constraint(m.i,m.s,m.h,rule=ec_repair_limit2)
    
def ec_restored_cost_limit(m,i):
    return m.ec_restored_cost[i] == sum(m.derepair_ec[i, s] * 50000 for s in ['M6', 'M7', 'M8'])
m.ec_restored_cost_limit = Constraint(m.i, rule=ec_restored_cost_limit)

m.ec_restored_Tcost = Var(domain=NonNegativeReals)
def ec_restored_cost_limit2(m):
    return m.ec_restored_Tcost == sum(m.ec_restored_cost[i] for i in m.i)
m.ec_restored_cost_limit2 = Constraint(rule=ec_restored_cost_limit2)

In [ ]:
m.b_hmax = Var(m.i,domain=NonNegativeReals)
m.b_onoff = Var(m.i,m.s,m.h,domain=Binary)
m.derepair_b = Var(m.i,m.s, domain=Binary)
m.b_restored_cost = Var(m.i,domain=NonNegativeReals)

def b_limit1(m,i,s,h):
    return m.b_hmax[i] <= m.pCap_max['boiler']
m.b_limit1 = Constraint(m.i,m.s,m.h,rule=b_limit1)
def b_limit2(m,i,s,h):
    return m.b_heat[i,s,h] <=  m.b_hmax[i]
m.b_limit2 = Constraint(m.i,m.s,m.h,rule=b_limit2)
def b_limit3(m):
    return m.b_hmax['b6'] == 0
m.b_limit3 = Constraint(rule=b_limit3)

def b_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.b_heat[i,s,h] <=m.pCap_max['boiler'] * 2
    elif Disaster_hour <= h_list1.index(h)+1 < 56:
        return m.b_heat[i,s,h] <= m.damage_b[i,s,h]* m.pCap_max['boiler'] * 2  # 修复期间，产能为0 
    else:
        return m.b_heat[i,s,h] <= m.damage_b[i,s,h] * m.pCap_max['boiler'] * 2 +m.derepair_b[i,s] * m.pCap_max['boiler'] * 2 #同时考虑设备损坏与修复
m.b_repair_limit2 = Constraint(m.i,m.s,m.h,rule=b_repair_limit2)
    

def b_restored_cost_limit(m,i):
    return m.b_restored_cost[i] == sum(m.derepair_b[i, s] * 20000 for s in ['M6', 'M7', 'M8'])
m.b_restored_cost_limit = Constraint(m.i, rule=b_restored_cost_limit)

m.b_restored_Tcost = Var(domain=NonNegativeReals)
def b_restored_cost_limit2(m):
    return m.b_restored_Tcost == sum(m.b_restored_cost[i] for i in m.i)
m.b_restored_cost_limit2 = Constraint(rule=b_restored_cost_limit2)

def b_not_run_limit2(m,i,h):
    return m.b_heat[i,'sum',h] == 0
m.b_not_run_limit2 = Constraint(m.i,m.h,rule=b_not_run_limit2)



In [ ]:
m.hp_hmax = Var(m.i,domain=NonNegativeReals)
m.hp_onoff = Var(m.i,m.s,m.h,domain=Binary)
m.derepair_hp = Var(m.i,m.s, domain=Binary)
m.hp_restored_cost = Var(m.i,domain=NonNegativeReals)

def hp_limit1(m,i,s,h):
    return m.hp_hmax[i] <=  m.pCap_max['hp']
m.hp_limit1 = Constraint(m.i,m.s,m.h,rule=hp_limit1)
def hp_limit2(m,i,s,h):
    return m.hp_heat[i,s,h] == m.hp_ele[i,s,h]*m.pEff['hp']
m.hp_limit2 = Constraint(m.i,m.s,m.h,rule=hp_limit2)

def hp_limit3(m,i,s,h):
    return m.hp_heat[i,s,h] <= m.hp_hmax[i]
m.hp_limit3 = Constraint(m.i,m.s,m.h,rule=hp_limit3)

def hp_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.hp_heat[i,s,h] <=m.pCap_max['hp'] * 2
    elif Disaster_hour <= h_list1.index(h)+1 < 108:
        return m.hp_heat[i,s,h] <= m.damage_hp[i,s,h]* m.pCap_max['hp'] * 2  
    else:
        return m.hp_heat[i,s,h] <= m.damage_hp[i,s,h] * m.pCap_max['hp'] * 2 +m.derepair_hp[i,s] * m.pCap_max['hp'] * 2 
m.hp_repair_limit2 = Constraint(m.i,m.s,m.h,rule=hp_repair_limit2)
    
def hp_restored_cost_limit(m,i):
    return m.hp_restored_cost[i] == sum(m.derepair_hp[i, s] * 50000 for s in ['M6', 'M7', 'M8'])
m.hp_restored_cost_limit = Constraint(m.i, rule=hp_restored_cost_limit)

m.hp_restored_Tcost = Var(domain=NonNegativeReals)
def hp_restored_cost_limit2(m):
    return m.hp_restored_Tcost == sum(m.hp_restored_cost[i] for i in m.i)
m.hp_restored_cost_limit2 = Constraint(rule=hp_restored_cost_limit2)

def hp_not_run_limit2(m,i,h):
   return m.hp_heat[i,'sum',h] == 0
m.hp_not_run_limit2 = Constraint(m.i,m.h,rule=hp_not_run_limit2)
def hp_not_run_limit3(m,i,h):
    return m.hp_heat[i,'M6',h] == 0
m.hp_not_run_limit3 = Constraint(m.i,m.h,rule=hp_not_run_limit3)
def hp_not_run_limit4(m,i,h):
    return m.hp_heat[i,'M7',h] == 0
m.hp_not_run_limit4 = Constraint(m.i,m.h,rule=hp_not_run_limit4)
def hp_not_run_limit5(m,i,h):
    return m.hp_heat[i,'M8',h] == 0
m.hp_not_run_limit5 = Constraint(m.i,m.h,rule=hp_not_run_limit5)



In [ ]:
m.ac_cmax = Var(m.i,domain=NonNegativeReals)
m.derepair_ac = Var(m.i,m.s, domain=Binary)
m.ac_restored_cost = Var(m.i,domain=NonNegativeReals)

def ac_limit1(m,i):
    return m.ac_cmax[i] <= m.pCap_max['ac']
m.ac_limit1 = Constraint(m.i,rule=ac_limit1)
def ac_limit2(m,i,s,h):
    return m.ac_cool[i,s,h] == m.ac_heat[i,s,h] * m.pEff['ac']
m.ac_limit2 = Constraint(m.i,m.s,m.h,rule=ac_limit2)
def ac_limit3(m,i,s,h):
    return m.ac_cool[i,s,h] <=  m.ac_cmax[i]
m.ac_limit3 = Constraint(m.i,m.s,m.h,rule=ac_limit3)
def ac_limit4(m):
    return m.ac_cmax['b6'] == 0
m.ac_limit4 = Constraint(rule=ac_limit4)

def ac_not_run_limit1(m,i,h):
    return m.ac_cool[i,'win',h] == 0
m.ac_not_run_limit1 = Constraint(m.i,m.h,rule=ac_not_run_limit1)
def ac_not_run_limit2(m,i,h):
    if i in ['b1', 'b2','b3', 'b4','b6']:
        return m.ac_cool[i,'mid',h] == 0
    else:
        return Constraint.Skip
m.ac_not_run_limit2 = Constraint(m.i,m.h,rule=ac_not_run_limit2)


def ac_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.ac_cool[i,s,h] <=m.pCap_max['ac'] * 2
    elif Disaster_hour <= h_list1.index(h)+1 < 132:
        return m.ac_cool[i,s,h] <= m.damage_ac[i,s,h]* m.pCap_max['ac'] * 2 
    else:
        return m.ac_heat[i,s,h] <= m.damage_ac[i,s,h] * m.pCap_max['ac'] * 2 +m.derepair_ac[i,s] * m.pCap_max['ac'] * 2 
m.ac_repair_limit2 = Constraint(m.i,m.s,m.h,rule=ac_repair_limit2)
    
def ac_restored_cost_limit(m,i):
    return m.ac_restored_cost[i] == sum(m.derepair_ac[i, s] * 50000 for s in ['M6', 'M7', 'M8'])
m.ac_restored_cost_limit = Constraint(m.i, rule=ac_restored_cost_limit)

m.ac_restored_Tcost = Var(domain=NonNegativeReals)
def ac_restored_cost_limit2(m):
    return m.ac_restored_Tcost == sum(m.ac_restored_cost[i] for i in m.i)
m.ac_restored_cost_limit2 = Constraint(rule=ac_restored_cost_limit2)

In [ ]:
m.pv_areamax = Var(m.i,domain=NonNegativeReals)
m.derepair_pv = Var(m.i,m.s, domain=Binary)
m.pv_restored_cost = Var(m.i,domain=NonNegativeReals)

def pv_limit1(m,i,s,h):
    return m.pv_e[i,s,h] <= m.pv_areamax[i]*m.pEff['pv']*(m.SRI[s,h]/1000)
m.pv_limit1 = Constraint(m.i,m.s,m.h,rule=pv_limit1)
def pv_limit2(m,i):
    return m.pv_areamax[i] <= m.pArea_roof[i]
m.pv_limit2 = Constraint(m.i,rule=pv_limit2)

def pv_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.pv_e[i,s,h] <=m.pv_areamax[i]*m.pEff['pv']*(m.SRI[s,h]/1000)
    elif Disaster_hour <= h_list1.index(h)+1 < 48:
        return m.pv_e[i,s,h] <= m.damage_pv[i,s,h]* m.pv_areamax[i]*m.pEff['pv']*(m.SRI[s,h]/1000)
    else:
        return m.pv_e[i,s,h] <= m.damage_pv[i,s,h] * m.pv_areamax[i]*m.pEff['pv']*(m.SRI[s,h]/1000) +m.derepair_pv[i,s] * m.pv_areamax[i]*m.pEff['pv']*(m.SRI[s,h]/1000) 
m.pv_repair_limit2 = Constraint(m.i,m.s,m.h,rule=pv_repair_limit2)
    
def pv_restored_cost_limit(m,i):
    return m.pv_restored_cost[i] == sum(m.derepair_pv[i, s] * 20000 for s in ['M6', 'M7', 'M8'])
m.pv_restored_cost_limit = Constraint(m.i, rule=pv_restored_cost_limit)

m.pv_restored_Tcost = Var(domain=NonNegativeReals)
def pv_restored_cost_limit2(m):
    return m.pv_restored_Tcost == sum(m.pv_restored_cost[i] for i in m.i)
m.pv_restored_cost_limit2 = Constraint(rule=pv_restored_cost_limit2)


In [ ]:
m.ele_st_max = Var(m.i,domain=NonNegativeReals)
m.ele_in_st = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.derepair_est = Var(m.i,m.s, domain=Binary)
m.est_restored_cost = Var(m.i,domain=NonNegativeReals)

def ele_st_limit2(m,i):
    return m.ele_st_max[i] <= m.pCap_max['ele_st']
m.ele_st_limit2 = Constraint(m.i,rule=ele_st_limit2)
### operating limit
def ele_st_limit3(m,i,s,h):
    return m.ele_in_st[i,s,h] <= m.ele_st_max[i]
m.ele_st_limit3 = Constraint(m.i,m.s,m.h,rule=ele_st_limit3)
def ele_st_limit4(m,i,s,h):
    return m.ele_in_st[i,s,h] == m.ele_in_st[i,s,h_1(h)]*m.pEff['ele_st'] + m.ele_cha[i,s,h]*m.pEff['ele_st'] - m.ele_dis[i,s,h]
m.ele_st_limit4 = Constraint(m.i,m.s,m.h,rule=ele_st_limit4)
def ele_st_limit5(m,i,s,h):
    return m.ele_cha[i,s,h] <= 0.5*m.ele_st_max[i]
m.ele_st_limit5 = Constraint(m.i,m.s,m.h,rule=ele_st_limit5)
def ele_st_limit6(m,i,s,h):
    return m.ele_dis[i,s,h] <= 0.5*m.ele_st_max[i]
m.ele_st_limit6 = Constraint(m.i,m.s,m.h,rule=ele_st_limit6)

def est_repair_limit1(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.ele_dis[i,s,h] <=0.5*m.ele_st_max[i]
    elif Disaster_hour <= h_list1.index(h)+1 < 84:
        return m.ele_dis[i,s,h] <= m.damage_ele_st[i,s,h] * 0.5*m.ele_st_max[i] 
    else:
        return m.ele_dis[i,s,h] <= m.damage_ele_st[i,s,h] * 0.5*m.ele_st_max[i] +m.derepair_est[i,s] * 0.5*m.ele_st_max[i] 
m.est_repair_limit1 = Constraint(m.i,m.s,m.h,rule=est_repair_limit1)

def est_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.ele_cha[i,s,h] <=0.5*m.ele_st_max[i]
    elif Disaster_hour <= h_list1.index(h)+1 < 84:
        return m.ele_cha[i,s,h] <= m.damage_ele_st[i,s,h] * 0.5*m.ele_st_max[i] 
    else:
        return m.ele_cha[i,s,h] <= m.damage_ele_st[i,s,h] * 0.5*m.ele_st_max[i] +m.derepair_est[i,s] * 0.5*m.ele_st_max[i]
m.est_repair_limit2 = Constraint(m.i,m.s,m.h,rule=est_repair_limit2)
    
def est_restored_cost_limit(m,i):
    return m.est_restored_cost[i] == sum(m.derepair_est[i, s] * 30000 for s in ['M6', 'M7', 'M8'])
m.est_restored_cost_limit = Constraint(m.i, rule=est_restored_cost_limit)

m.est_restored_Tcost = Var(domain=NonNegativeReals)
def est_restored_cost_limit2(m):
    return m.est_restored_Tcost == sum(m.est_restored_cost[i] for i in m.i)
m.est_restored_cost_limit2 = Constraint(rule=est_restored_cost_limit2)


In [ ]:
m.cool_st_max = Var(m.i,domain=NonNegativeReals)
m.cool_in_st = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.derepair_cst = Var(m.i,m.s, domain=Binary)
m.cst_restored_cost = Var(m.i,domain=NonNegativeReals)
m.c_cha = Var(m.i,m.s,m.h, domain=Binary)
m.c_dis = Var(m.i,m.s,m.h, domain=Binary)

def cool_st_limit2(m,i):
    return m.cool_st_max[i] <= m.pCap_max['cool_st']
m.cool_st_limit2 = Constraint(m.i,rule=cool_st_limit2)
### operating limit
def cool_st_limit3(m,i,s,h):
    return m.cool_in_st[i,s,h] <= m.cool_st_max[i]
m.cool_st_limit3 = Constraint(m.i,m.s,m.h,rule=cool_st_limit3)
def cool_st_limit4(m,i,s,h):
    return m.cool_in_st[i,s,h] == m.cool_in_st[i,s,h_1(h)]*m.pEff['cool_st'] + m.cool_cha[i,s,h]*m.pEff['cool_st'] - m.cool_dis[i,s,h]
m.cool_st_limit4 = Constraint(m.i,m.s,m.h,rule=cool_st_limit4)
def cool_st_limit5(m,i,s,h):
    return m.cool_cha[i,s,h] <= 0.5*m.cool_st_max[i] 
m.cool_st_limit5 = Constraint(m.i,m.s,m.h,rule=cool_st_limit5)
def cool_st_limit6(m,i,s,h):
    return m.cool_dis[i,s,h] <= 0.5*m.cool_st_max[i] 
m.cool_st_limit6 = Constraint(m.i,m.s,m.h,rule=cool_st_limit6)
def cool_st_limit7(m):
    return m.cool_st_max['b6'] == 0
m.cool_st_limit7 = Constraint(rule=cool_st_limit7)

def cst_not_run_limit1(m,i,h):
    return m.cool_cha[i,'win',h] == 0
m.cst_not_run_limit1 = Constraint(m.i,m.h,rule=cst_not_run_limit1)
def cst_not_run_limit2(m,i,h):
    return m.cool_dis[i,'win',h] == 0
m.cst_not_run_limit2 = Constraint(m.i,m.h,rule=cst_not_run_limit2)
def cst_not_run_limit3(m,i,h):
    if i in ['b1', 'b2','b3', 'b4','b6']:
       return m.cool_dis[i,'mid',h] == 0   
    else:
        return Constraint.Skip
m.cst_not_run_limit3 = Constraint(m.i,m.h,rule=cst_not_run_limit3)
def cst_not_run_limit4(m,i,h):
    if i in ['b1', 'b2','b3', 'b4','b6']:
       return m.cool_cha[i,'mid',h] == 0  
    else:
        return Constraint.Skip
m.cst_not_run_limit4 = Constraint(m.i,m.h,rule=cst_not_run_limit4)

def cst_repair_limit1(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.cool_dis[i,s,h] <=0.5*m.cool_st_max[i]
    elif Disaster_hour <= h_list1.index(h)+1 < 108:
        return m.cool_dis[i,s,h] <= m.damage_cool_st[i,s,h] * 0.5*m.cool_st_max[i]
    else:
        return m.cool_dis[i,s,h] <= m.damage_cool_st[i,s,h] * 0.5*m.cool_st_max[i] +m.derepair_cst[i,s] * 0.5*m.cool_st_max[i]
m.cst_repair_limit1 = Constraint(m.i,m.s,m.h,rule=cst_repair_limit1)

def cst_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.cool_cha[i,s,h] <=0.5*m.cool_st_max[i]
    elif Disaster_hour <= h_list1.index(h)+1 < 108:
        return m.cool_cha[i,s,h] <= m.damage_cool_st[i,s,h] * 0.5*m.cool_st_max[i] 
    else:
        return m.cool_cha[i,s,h] <= m.damage_cool_st[i,s,h] * 0.5*m.cool_st_max[i] +m.derepair_cst[i,s] * 0.5*m.cool_st_max[i]
m.cst_repair_limit2 = Constraint(m.i,m.s,m.h,rule=cst_repair_limit2)

def cst_restored_cost_limit(m,i):
    return m.cst_restored_cost[i] == sum(m.derepair_cst[i, s] * 30000 for s in ['M6', 'M7', 'M8'])
m.cst_restored_cost_limit = Constraint(m.i, rule=cst_restored_cost_limit)

m.cst_restored_Tcost = Var(domain=NonNegativeReals)
def cst_restored_cost_limit2(m):
    return m.cst_restored_Tcost == sum(m.cst_restored_cost[i] for i in m.i)
m.cst_restored_cost_limit2 = Constraint(rule=cst_restored_cost_limit2)

In [ ]:
m.derepair_grid = Var(m.i,m.s, domain=Binary)
m.grid_restored_cost = Var(m.i,domain=NonNegativeReals)

def grid_limit1(m,i,s,h):
    return m.grid_ex[i,s,h] <= m.pCap_max['grid']
m.grid_limit1 = Constraint(m.i,m.s,m.h,rule=grid_limit1)
def grid_limit2(m,i,s,h):
    return m.grid_im[i,s,h] <= m.pCap_max['grid']
m.grid_limit2 = Constraint(m.i,m.s,m.h,rule=grid_limit2)
### 22-6点不向电网卖电
def grid_sell_limit(m,i,s,h):
    if h_list.index(h)+1 <= 7 or h_list.index(h)+1 >= 22: return m.grid_ex[i,s,h] == 0
    else: return m.grid_ex[i,s,h] >= 0
m.grid_sell_limit = Constraint(m.i,m.s,m.h,rule=grid_sell_limit)

def grid_sell_limit2(m,s,h):
    return m.grid_ex['b6',s,h]== 0
m.grid_sell_limit2 = Constraint(m.s,m.h,rule=grid_sell_limit2)

def grid_repair_limit1(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.grid_ex[i,s,h] <= m.pCap_max['grid']
    elif Disaster_hour <= h_list1.index(h)+1 < 48:
        return m.grid_ex[i,s,h] <= m.damage_grid[i,s,h] *m.pCap_max['grid'] 
    else:
        return m.grid_ex[i,s,h] <= m.damage_grid[i,s,h] *m.pCap_max['grid'] +m.derepair_grid[i,s] *m.pCap_max['grid'] 

def grid_repair_limit2(m,i,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.grid_im[i,s,h] <=m.pCap_max['grid']
    elif Disaster_hour <= h_list1.index(h)+1 < 48:
        return m.grid_im[i,s,h] <= m.damage_grid[i,s,h] *m.pCap_max['grid'] 
    else:
        return m.grid_im[i,s,h] <= m.damage_grid[i,s,h] *m.pCap_max['grid'] +m.derepair_grid[i,s] *m.pCap_max['grid'] 
m.grid_repair_limit2 = Constraint(m.i,m.s,m.h,rule=grid_repair_limit2)
    
def grid_restored_cost_limit(m,i):
    return m.grid_restored_cost[i] == sum(m.derepair_grid[i, s] * 10000 for s in ['M6', 'M7', 'M8'])
m.grid_restored_cost_limit = Constraint(m.i, rule=grid_restored_cost_limit)

m.grid_restored_Tcost = Var(domain=NonNegativeReals)
def grid_restored_cost_limit2(m):
    return m.grid_restored_Tcost == sum(m.grid_restored_cost[i] for i in m.i)
m.grid_restored_cost_limit2 = Constraint(rule=grid_restored_cost_limit2)

In [ ]:
m.yTr_h = Var(m.i,m.j,m.s,m.h,domain=Binary)
m.yPipe_h = Var(m.i,m.j,domain=Binary)
m.derepair_hpipe = Var(m.i,m.s, domain=Binary)
m.hpipe_restored_cost = Var(m.i,domain=NonNegativeReals)

def tr_h_limit1(m,i,j,s,h):
    return m.tr_h[i,j,s,h] <= m.yTr_h[i,j,s,h]*m.pCap_max['boiler']
m.tr_h_limit1 = Constraint(m.i,m.j,m.s,m.h,rule=tr_h_limit1)
def tr_h_limit2(m,i,j,s,h):
    return m.yTr_h[i,j,s,h] + m.yTr_h[j,i,s,h] <= 1
m.tr_h_limit2 = Constraint(m.i,m.j,m.s,m.h,rule=tr_h_limit2)

# tr_h.fx(i,j,'sum',d,h) =0; tr_h.fx(i,j,'mid',d,h) =0;
for i in m.i:
    for j in m.j:
        for h in m.h:
            m.yTr_h[i,j,'sum',h].fix(0)
            m.yTr_h[i,j,'mid',h].fix(0)
# m.vH_tr['b1','b1','sum','workday','h1'].pprint()

indices = [['b1','b1'],['b2','b2'],['b3','b3'],['b4','b4'],['b5','b5'],['b6','b6'],['b1','b3'],['b1','b4'],['b2','b3'],['b3','b1'],['b3','b2'],['b3','b6'],['b4','b1'],['b4','b6'],['b5','b6'],['b6','b3'],['b6','b4'],['b6','b5']]
for s in m.s:
        for h in m.h:
            for index in indices:
                m.tr_h[index[0], index[1], s, h].fix(0)

def tr_h_limit3(m,i,j):
    return m.yPipe_h[i,j] == m.yPipe_h[j,i]
m.tr_h_limit3 = Constraint(m.i,m.j,rule=tr_h_limit3)

for s in m.s:
        for h in m.h:
            for index in indices:
                m.yPipe_h[index[0], index[1]].fix(0)

def tr_h_limit4(m,i,j,s,h):
    return m.tr_h[i,j,s,h] <= m.yPipe_h[i,j]*m.pCap_max['chp']
m.tr_h_limit4 = Constraint(m.i,m.j,m.s,m.h,rule=tr_h_limit4)

 
def tr_h_repair_limit2(m,i,j,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.tr_h[i,j,s,h] <=m.pCap_max['chp'] 
    elif Disaster_hour <= h_list1.index(h)+1 < 48:
        return m.tr_h[i,j,s,h] <= m.damage_pipe[i,j,s,h]* m.pCap_max['chp']  
    else:
        return m.tr_h[i,j,s,h] <= m.damage_pipe[i,j,s,h] * m.pCap_max['chp']  +m.derepair_hpipe[i,s] * m.pCap_max['chp']  
m.tr_h_repair_limit2 = Constraint(m.i,m.j,m.s,m.h,rule=tr_h_repair_limit2)
    
def tr_h_restored_cost_limit(m,i):
    return m.hpipe_restored_cost[i] == sum(m.derepair_hpipe[i, s] * 20000 for s in ['M6', 'M7', 'M8'])
m.tr_h_restored_cost_limit = Constraint(m.i, rule=tr_h_restored_cost_limit)

m.hpipe_restored_Tcost = Var(domain=NonNegativeReals)
def hpipe_restored_cost_limit2(m):
    return m.hpipe_restored_Tcost == sum(m.hpipe_restored_cost[i] for i in m.i)
m.hpipe_restored_cost_limit2 = Constraint(rule=hpipe_restored_cost_limit2)


In [ ]:
m.yTr_c = Var(m.i,m.j,m.s,m.h,domain=Binary)
m.yPipe_c = Var(m.i,m.j,domain=Binary)
m.derepair_cpipe = Var(m.i,m.s, domain=Binary)
m.cpipe_restored_cost = Var(m.i,domain=NonNegativeReals)

def tr_c_limit1(m,i,j,s,h):
    return m.tr_c[i,j,s,h] <= m.yTr_c[i,j,s,h]*m.pCap_max['ec']
m.tr_c_limit1 = Constraint(m.i,m.j,m.s,m.h,rule=tr_c_limit1)
def tr_c_limit2(m,i,j,s,h):
    return m.yTr_c[i,j,s,h] + m.yTr_c[j,i,s,h] <= 1
m.tr_c_limit2 = Constraint(m.i,m.j,m.s,m.h,rule=tr_c_limit2)

for i in m.i:
    for j in m.j:
        for h in m.h:
            m.yTr_c[i,j,'sum',h].fix(0)
            m.yTr_c[i,j,'mid',h].fix(0)

indices = [['b1','b1'],['b2','b2'],['b3','b3'],['b4','b4'],['b5','b5'],['b6','b6'],['b1','b3'],['b1','b4'],['b2','b3'],['b3','b1'],['b3','b2'],['b3','b6'],['b4','b1'],['b4','b6'],['b5','b6'],['b6','b3'],['b6','b4'],['b6','b5']]

for s in m.s:
     for h in m.h:
        for index in indices:
            m.tr_c[index[0], index[1], s, h].fix(0)

def tr_c_limit3(m,i,j):
    return m.yPipe_c[i,j] == m.yPipe_c[j,i]
m.tr_c_limit3 = Constraint(m.i,m.j,rule=tr_c_limit3)

for s in m.s:
      for h in m.h:
            for index in indices:
                m.yPipe_c[index[0], index[1]].fix(0)
def tr_c_limit4(m,i,j,s,h):
    return m.tr_c[i,j,s,h] <= m.yPipe_c[i,j]*m.pCap_max['ec']
m.tr_c_limit4 = Constraint(m.i,m.j,m.s,m.h,rule=tr_c_limit4)


def tr_c_repair_limit2(m,i,j,s,h):
    if h_list1.index(h)+1 < Disaster_hour:
        return m.tr_c[i,j,s,h] <=m.pCap_max['ec'] * 2
    elif Disaster_hour <= h_list1.index(h)+1 < 48:
        return m.tr_c[i,j,s,h] <= m.damage_pipe[i,j,s,h]* m.pCap_max['ec']  
    else:
        return m.tr_c[i,j,s,h] <= m.damage_pipe[i,j,s,h] * m.pCap_max['ec']  +m.derepair_cpipe[i,s] * m.pCap_max['ec']  
m.tr_c_repair_limit2 = Constraint(m.i,m.j,m.s,m.h,rule=tr_c_repair_limit2)
    
def tr_c_restored_cost_limit(m,i):
    return m.cpipe_restored_cost[i] == sum(m.derepair_cpipe[i, s] * 20000 for s in ['M6', 'M7', 'M8'])
m.tr_c_restored_cost_limit = Constraint(m.i, rule=tr_c_restored_cost_limit)

m.cpipe_restored_Tcost = Var(domain=NonNegativeReals)
def cpipe_restored_cost_limit2(m):
    return m.cpipe_restored_Tcost == sum(m.cpipe_restored_cost[i] for i in m.i)
m.cpipe_restored_cost_limit2 = Constraint(rule=cpipe_restored_cost_limit2)


In [ ]:
m.ele_output = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.ele_loss = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.heat_output = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.heat_loss = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.cool_output = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.cool_loss = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.Total_Load = Var(m.s,domain=NonNegativeReals)
m.load_loss = Var(m.s,domain=NonNegativeReals)
m.Totalloss_c = Var(m.s,domain=NonNegativeReals)
m.Disaster_cost = Var(domain=NonNegativeReals)
m.restored_cost = Var(domain=NonNegativeReals)
m.EENS = Var(domain=NonNegativeReals)
m.EIU = Var(domain=NonNegativeReals)

def Total_load_limit(m,s):
    return m.Total_Load[s] == sum( m.e_demand[i,s,h]+(m.h_demand[i,s,h]/3)+(m.c_demand[i,s,h]/4) for i in m.i for h in m.h)
m.Total_load_limit = Constraint(m.s,rule=Total_load_limit)

def ele_output_limit(m,i,s,h):
    return m.ele_output[i,s,h] == m.ele_dis[i,s,h] + m.CHP_e[i,s,h] + m.grid_im[i,s,h] + m.pv_e[i,s,h] - m.ec_ele[i,s,h] - m.hp_ele[i,s,h] - m.grid_ex[i,s,h] - m.ele_cha[i,s,h]
m.ele_output_limit = Constraint(m.i,m.s,m.h,rule=ele_output_limit)

def heat_output_limit(m,i,s,h):
    return m.heat_output[i,s,h] == m.hp_heat[i,s,h] + m.CHP_e[i,s,h] *m.pH_to_P + m.b_heat[i,s,h]  + sum(m.tr_h[j,i,s,h]*(1-m.pLoss_rate_heat_pipe) for j in m.j) - sum(m.tr_h[j,i,s,h] for j in m.j) - m.ac_heat[i,s,h]
m.heat_output_limit = Constraint(m.i,m.s,m.h,rule=heat_output_limit)

def cool_output_limit(m,i,s,h):
    return m.cool_output[i,s,h] == m.ec_cool[i,s,h] + m.ac_cool[i,s,h] + sum(m.tr_c[j,i,s,h]*(1-m.pLoss_rate_cool_pipe) for j in m.j) +m.cool_dis[i,s,h] - sum(m.tr_c[j,i,s,h] for j in m.j)-m.cool_cha[i,s,h]
m.cool_output_limit = Constraint(m.i,m.s,m.h,rule=cool_output_limit)

def ele_loss_limit(m,i,s,h): 
    return m.ele_loss[i,s,h] == m.e_demand[i,s,h] - m.ele_output[i,s,h]
m.ele_loss_limit = Constraint(m.i,m.s,m.h,rule=ele_loss_limit)

def heat_loss_limit(m,i,s,h):
    return m.heat_loss[i,s,h] == m.h_demand[i,s,h] - m.heat_output[i,s,h]
m.heat_loss_limit = Constraint(m.i,m.s,m.h,rule=heat_loss_limit)

def cool_loss_limit(m,i,s,h):
    return m.cool_loss[i,s,h] == m.c_demand[i,s,h] - m.cool_output[i,s,h]
m.cool_loss_limit = Constraint(m.i,m.s,m.h,rule=cool_loss_limit)
def load_loss_limit(m,s):
    return m.load_loss[s] == sum(m.ele_loss[i,s,h]+(m.heat_loss[i,s,h]/3)+(m.cool_loss[i,s,h]/4) for i in m.i for h in m.h)
m.load_loss_limit = Constraint(m.s,rule=load_loss_limit)

def Totalloss_c_limit(m,s):
    return m.Totalloss_c[s] == m.load_loss[s] * 1800
m.Totalloss_c_limit = Constraint(m.s,rule=Totalloss_c_limit)
def Disaster_cost_limit(m):
    return m.Disaster_cost == sum(m.Pro[s]*m.Totalloss_c[s]*m.qty_day[s,i] for s in m.s)
m.Disaster_cost_limit = Constraint(rule=Disaster_cost_limit)

def eens_limit(m):
    return m.EENS == sum(m.Pro[s]*m.load_loss[s] for s in m.s)
m.eens_limit = Constraint(rule=eens_limit)
def eiu_limit(m):
    return m.EIU == m.EENS/3372805
m.eiu_limit = Constraint(rule=eiu_limit)


def restored_cost_limit(m):
    return m.restored_cost == sum((m.chp_restored_cost[i]+m.ec_restored_cost[i]+m.b_restored_cost[i]+m.hp_restored_cost[i]+m.ac_restored_cost[i]+m.pv_restored_cost[i]+m.est_restored_cost[i]+m.cst_restored_cost[i]+m.grid_restored_cost[i]+m.hpipe_restored_cost[i]+m.cpipe_restored_cost[i]) for i in m.i)
m.restored_cost_limit = Constraint(rule=restored_cost_limit)


In [ ]:
m.vCC_device = Var(m.i,domain=NonNegativeReals)
m.vCC_device_annualized = Var(domain=NonNegativeReals)

def cc_device_limit(m,i):
    return m.vCC_device[i] == m.CHP_emax[i]*m.pCost_unit['chp']+m.ec_cmax[i]*m.pCost_unit['ec']+m.b_hmax[i]*m.pCost_unit['boiler']+m.hp_hmax[i]*m.pCost_unit['hp']+m.ac_cmax[i]*m.pCost_unit['ac']+m.pv_areamax[i]/6*m.pCost_unit['pv']+m.cool_st_max[i]*m.pCost_unit['cool_st']+m.ele_st_max[i]*m.pCost_unit['ele_st']
m.cc_device_limit = Constraint(m.i,rule=cc_device_limit)
def cc_device_limit2(m):
    return m.vCC_device_annualized == sum(m.vCC_device[i]*m.CRF1 for i in m.i)
m.cc_device_limit2 = Constraint(rule=cc_device_limit2)


m.vCC_pipe = Var(domain=NonNegativeReals)
m.vCC_pipe_annualized = Var(domain=NonNegativeReals)
def cc_pipe_limit1(m):
    return m.vCC_pipe == m.pCost_unit_hpipe*sum(m.yPipe_h[i,j]*m.dist[i,j] for i in m.i for j in m.j)/2 + m.pCost_unit_cpipe*sum(m.yPipe_c[i,j]*m.dist[i,j] for i in m.i for j in m.j)/2
m.cc_pipe_limit1 = Constraint(rule=cc_pipe_limit1)
def cc_pipe_limit2(m):
    return m.vCC_pipe_annualized == m.vCC_pipe*m.CRF2
m.cc_pipe_limit2 = Constraint(rule=cc_pipe_limit2)

m.vCC_fuel = Var(m.i,domain=NonNegativeReals)
m.vCC_fuel_total = Var(domain=NonNegativeReals)
def cc_fuel_limit1(m,i):
    return m.vCC_fuel[i] == sum(m.Pro[s]*m.qty_day[s,i]*(sum((m.CHP_e[i,s,h]/m.pEff['chp']+m.b_heat[i,s,h]/m.pEff['boiler'])*m.price['NG',h] for h in m.h)) for s in m.s )
m.cc_fuel_limit1 = Constraint(m.i,rule=cc_fuel_limit1)
def cc_fuel_limit2(m):
    return m.vCC_fuel_total == sum(m.vCC_fuel[i] for i in m.i)
m.cc_fuel_limit2 = Constraint(rule=cc_fuel_limit2)

m.vCC_grid_im = Var(m.i,domain=NonNegativeReals)
m.vCC_grid_im_total = Var(domain=NonNegativeReals)
def cc_grid_limit1(m,i):
    if i =='b6':
        return m.vCC_grid_im[i] == sum(m.Pro[s]*m.qty_day[s,i]*(sum(m.grid_im[i,s,h]*m.price['grid_buy2',h] for h in m.h)) for s in m.s )
    else:
        return m.vCC_grid_im[i] == sum(m.Pro[s]*m.qty_day[s,i]*(sum(m.grid_im[i,s,h]*m.price['grid_buy1',h] for h in m.h)) for s in m.s )
m.cc_grid_limit1 = Constraint(m.i,rule=cc_grid_limit1)
def cc_grid_limit2(m):
    return m.vCC_grid_im_total == sum(m.vCC_grid_im[i] for i in m.i)
m.cc_grid_limit2 = Constraint(rule=cc_grid_limit2)

m.vCC_grid_ex = Var(m.i,domain=NonNegativeReals)
m.vCC_grid_ex_total = Var(domain=NonNegativeReals)
def cc_grid_limit3(m,i):
    return m.vCC_grid_ex[i] == sum(m.Pro[s]*m.qty_day[s,i]*(sum(m.grid_ex[i,s,h]*m.price['grid_sell',h] for h in m.h)) for s in m.s )
m.cc_grid_limit3 = Constraint(m.i,rule=cc_grid_limit3)
def cc_grid_limit4(m):
    return m.vCC_grid_ex_total == sum(m.vCC_grid_ex[i] for i in m.i)
m.cc_grid_limit4 = Constraint(rule=cc_grid_limit4)

m.vCC_maint_houly = Var(m.i,m.s,m.h,domain=NonNegativeReals)
m.vCC_maint = Var(m.i,domain=NonNegativeReals)
m.vCC_maint_total = Var(domain=NonNegativeReals)
def cc_maint_limit1(m,i,s,h):
    return m.vCC_maint_houly[i,s,h] == m.CHP_e[i,s,h]*m.pMaint['chp']+m.ec_cool[i,s,h]*m.pMaint['ec']+m.b_heat[i,s,h]*m.pMaint['boiler']+m.hp_heat[i,s,h]*m.pMaint['hp']+m.ac_cool[i,s,h]*m.pMaint['ac']+m.pv_e[i,s,h]*m.pMaint['pv']+m.ele_in_st[i,s,h]*m.pMaint['ele_st']+m.cool_in_st[i,s,h]*m.pMaint['cool_st']
m.cc_maint_limit1 = Constraint(m.i,m.s,m.h,rule=cc_maint_limit1)
def cc_maint_limit2(m,i):
    return m.vCC_maint[i] == sum(m.Pro[s]*m.qty_day[s,i]*(sum(m.vCC_maint_houly[i,s,h] for h in m.h)) for s in m.s )
m.cc_maint_limit2 = Constraint(m.i,rule=cc_maint_limit2)
def cc_maint_limit3(m):
    return m.vCC_maint_total == sum(m.vCC_maint[i] for i in m.i)
m.cc_maint_limit3 = Constraint(rule=cc_maint_limit3)

m.obj_TAC = Var(domain=NonNegativeReals)
def TAC_limit(m):
    return m.obj_TAC == m.vCC_device_annualized + m.vCC_pipe_annualized + m.vCC_fuel_total + m.vCC_grid_im_total - m.vCC_grid_ex_total + m.vCC_maint_total + m.Disaster_cost + m.restored_cost
m.TAC_limit = Constraint(rule=TAC_limit)

## 7. Objective function

Minimize the model total annualized cost (`obj_TAC`), which combines device and network capital cost, fuel, grid exchange, maintenance, unmet-demand cost, and restoration cost.


In [ ]:
def obj(m):
    return m.obj_TAC
m.obj = Objective(rule=obj, sense=minimize, doc='选定目标函数')

## 8. Solve and collect results

Run the Gurobi optimization for each Monte Carlo realization and copy objective, cost, resilience, capacity, restoration, and hourly dispatch values into the in-memory `m.ag_*` result containers.


In [ ]:

solver = SolverFactory('gurobi')
mipgap = 0.001
tmlim = 2400
solver.options['MIPGap'] = mipgap
solver.options['TimeLimit'] = tmlim

for sr in m.sr:
    for i in m.i:
        for s in m.s:
            m.damage_hour_chp1[i,s] = m.damage_hour_chp[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_boiler1[i,s] = m.damage_hour_boiler[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_ec1[i,s] = m.damage_hour_ec[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_ac1[i,s] = m.damage_hour_ac[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_hp1[i,s] = m.damage_hour_hp[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_cool_st1[i,s] = m.damage_hour_cool_st[sr,i,s]
    # for i in m.i:
    #     for s in m.s:
    #         m.damage_hour_heat_st1[i,s] = m.damage_hour_heat_st[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_ele_st1[i,s] = m.damage_hour_ele_st[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_grid1[i,s] = m.damage_hour_grid[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_pv1[i,s] = m.damage_hour_pv[sr,i,s]
    for i in m.i:
        for s in m.s:
            m.damage_hour_pipe1[i,s] = m.damage_hour_pipe[sr,i,s]
        
    for i in m.i:  
        for s in m.s:  
            for h in m.h: 
                num_chp = 0 if m.damage_hour_chp1[i,s].value <= h_list1.index(h)+1  else 1
                num_b = 0 if m.damage_hour_boiler1[i,s].value <= h_list1.index(h)+1  else 1
                num_hp = 0 if m.damage_hour_hp1[i,s].value <= h_list1.index(h)+1  else 1
                num_pv = 0 if m.damage_hour_pv1[i,s].value <= h_list1.index(h)+1  else 1
                num_ec = 0 if m.damage_hour_ec1[i,s].value <= h_list1.index(h)+1  else 1
                num_ac = 0 if m.damage_hour_ac1[i,s].value <= h_list1.index(h)+1  else 1
                num_ele_st = 0 if m.damage_hour_ele_st1[i,s].value <= h_list1.index(h)+1  else 1
                num_cool_st = 0 if m.damage_hour_cool_st1[i,s].value <= h_list1.index(h)+1  else 1
                num_grid = 0 if m.damage_hour_grid1[i,s].value <= h_list1.index(h)+1  else 1

                # num_chp = 1 if m.damage_hour_chp1[i,s].value >= h_list1.index(h)+1 else 0
                # num_b = 1 if m.damage_hour_boiler1[i,s].value >= h_list1.index(h)+1 else 0  
                # num_hp = 1 if m.damage_hour_hp1[i,s].value >= h_list1.index(h)+1 else 0  
                # num_pv = 1 if m.damage_hour_pv1[i,s].value >= h_list1.index(h)+1 else 0  
                # num_ec = 1 if m.damage_hour_ec1[i,s].value >= h_list1.index(h)+1 else 0   
                # num_ac = 1 if m.damage_hour_ac1[i,s].value >= h_list1.index(h)+1 else 0  
                # num_ele_st = 1 if m.damage_hour_ele_st1[i,s].value >= h_list1.index(h)+1 else 0  
                # # num_heat_st = 1 if m.damage_hour_heat_st1[i,s].value >= h_list1.index(h)+1 else 0  
                # num_cool_st = 1 if m.damage_hour_cool_st1[i,s].value >= h_list1.index(h)+1 else 0  
                # num_grid = 1 if m.damage_hour_grid1[i,s].value >= h_list1.index(h)+1 else 0

                m.damage_chp[i,s,h] = num_chp
                m.damage_b[i,s,h] = num_b  
                m.damage_hp[i,s,h] = num_hp  
                m.damage_pv[i,s,h] = num_pv  
                m.damage_ec[i,s,h] = num_ec  
                m.damage_ac[i,s,h] = num_ac  
                m.damage_ele_st[i,s,h] = num_ele_st  
                # m.damage_heat_st[i,s,h] = num_heat_st  
                m.damage_cool_st[i,s,h] = num_cool_st  
                m.damage_grid[i,s,h] = num_grid    
    for i in m.i: 
        for j in m.j: 
            for s in m.s:  
                for h in m.h:     
                   num_pipe = 0 if m.damage_hour_pipe1[i,s].value <= h_list1.index(h)+1 < m.damage_hour_pipe1[i,s].value+24 else 1       
                   m.damage_pipe[i,j,s,h] = num_pipe 
                
   
    
    results = solver.solve(m, tee=True)
    if results.solver.termination_condition == TerminationCondition.infeasible:
         continue
    

    obj_TAC = m.obj_TAC.value
    m.ag_results[sr] = obj_TAC
    Disaster_cost = m.Disaster_cost.value
    m.ag_Disaster_cost[sr] = Disaster_cost
    device_cost = m.vCC_device_annualized.value
    m.ag_device_cost[sr] = device_cost
    pipe_cost = m.vCC_pipe_annualized.value
    m.ag_pipe_cost[sr] = pipe_cost
    maint_cost = m.vCC_maint_total
    m.ag_maint_cost[sr] = maint_cost
    fuel_cost = m.vCC_fuel_total
    m.ag_fuel_cost[sr] = fuel_cost
    grid_im_cost = m.vCC_grid_im_total
    m.ag_grid_im_cost[sr] = grid_im_cost
    grid_ex_cost = m.vCC_grid_ex_total
    m.ag_grid_ex_cost[sr] = grid_ex_cost
    restored_cost = m.restored_cost
    m.ag_restored_cost[sr] = restored_cost
    EENS = m.EENS.value
    m.ag_EENS[sr] = EENS
    EIU = m.EIU.value
    m.ag_EIU[sr] = EIU

    chp_restored_cost = m.chp_restored_Tcost
    m.ag_chp_restored_Tcost[sr] = chp_restored_cost
    ec_restored_cost = m.ec_restored_Tcost
    m.ag_ec_restored_Tcost[sr] = ec_restored_cost
    b_restored_cost = m.b_restored_Tcost
    m.ag_b_restored_Tcost[sr] = b_restored_cost
    hp_restored_cost = m.hp_restored_Tcost
    m.ag_hp_restored_Tcost[sr] = hp_restored_cost
    ac_restored_cost = m.ac_restored_Tcost
    m.ag_ac_restored_Tcost[sr] = ac_restored_cost
    pv_restored_cost = m.pv_restored_Tcost
    m.ag_pv_restored_Tcost[sr] = pv_restored_cost
    est_restored_cost = m.est_restored_Tcost
    m.ag_est_restored_Tcost[sr] = est_restored_cost
    cst_restored_cost = m.cst_restored_Tcost
    m.ag_cst_restored_Tcost[sr] = cst_restored_cost
    grid_restored_cost = m.grid_restored_Tcost
    m.ag_grid_restored_Tcost[sr] = grid_restored_cost
    hpipe_restored_cost = m.hpipe_restored_Tcost
    m.ag_hpipe_restored_Tcost[sr] = hpipe_restored_cost
    cpipe_restored_cost = m.cpipe_restored_Tcost
    m.ag_cpipe_restored_Tcost[sr] = cpipe_restored_cost


    for i in m.i:
        for j in m.j:
            for s in m.s:
                for h in m.h:
                    chp_e = m.CHP_e[i, s, h].value
                    m.ag_chp_e[sr,i,s,h] = chp_e
                    pv_e = m.pv_e[i,s,h].value
                    m.ag_pv_e[sr,i,s,h] = pv_e
                    ec_ele = m.ec_ele[i,s,h].value
                    m.ag_ec_ele[sr,i,s,h] = ec_ele
                    ec_cool = m.ec_cool[i,s,h].value
                    m.ag_ec_cool[sr,i,s,h] = ec_cool
                    hp_ele = m.hp_ele[i,s,h].value
                    m.ag_hp_ele[sr,i,s,h] = hp_ele
                    hp_heat = m.hp_heat[i,s,h].value
                    m.ag_hp_heat[sr,i,s,h] = hp_heat
                    b_heat = m.b_heat[i,s,h].value
                    m.ag_b_heat[sr,i,s,h] = b_heat
                    ac_heat = m.ac_heat[i,s,h].value
                    m.ag_ac_heat[sr,i,s,h] = ac_heat
                    ac_cool = m.ac_cool[i,s,h].value
                    m.ag_ac_cool[sr,i,s,h] = ac_cool
                    ele_cha = m.ele_cha[i, s, h].value
                    m.ag_ele_cha[sr,i,s,h] = ele_cha
                    ele_dis = m.ele_dis[i, s, h].value
                    m.ag_ele_dis[sr,i,s,h] = ele_dis
                    cool_cha = m.cool_cha[i, s, h].value
                    m.ag_cool_cha[sr,i,s,h] = cool_cha
                    cool_dis = m.cool_dis[i, s, h].value
                    m.ag_cool_dis[sr,i,s,h] = cool_dis
                    grid_im = m.grid_im[i, s, h].value
                    m.ag_grid_im[sr,i,s,h] = grid_im
                    grid_ex = m.grid_ex[i, s, h].value
                    m.ag_grid_ex[sr,i,s,h] = grid_ex
                    tr_h = m.tr_h[i,j, s, h].value
                    m.ag_tr_h[sr,i,j,s,h] = tr_h
                    tr_c = m.tr_c[i,j, s, h].value
                    m.ag_tr_c[sr,i,j,s,h] = tr_c
                    derepair_chp = m.derepair_chp[i,s].value
                    m.ag_derepair_chp[sr,i,s] = derepair_chp
                    
                    

                    
                    

                    CHP_emax = m.CHP_emax[i].value
                    m.ag_CHP_emax[sr,i] = CHP_emax
                    ec_cmax = m.ec_cmax[i].value
                    m.ag_ec_cmax[sr,i] = ec_cmax
                    ac_cmax = m.ac_cmax[i].value
                    m.ag_ac_cmax[sr,i] = ac_cmax
                    hp_hmax = m.hp_hmax[i].value
                    m.ag_hp_hmax[sr,i] = hp_hmax
                    b_hmax = m.b_hmax[i].value
                    m.ag_b_hmax[sr,i] = b_hmax
                    ele_st_max = m.ele_st_max[i].value
                    m.ag_ele_st_max[sr,i] = ele_st_max
                    cool_st_max = m.cool_st_max[i].value
                    m.ag_cool_st_max[sr,i] = cool_st_max
                    pv_areamax = m.pv_areamax[i].value
                    m.ag_pv_areamax[sr,i] = pv_areamax
        
